# Verify

Check that the demo is deployed and that a moment turns into a strategy call. This does the same as `scripts/verify.sh`: **Expect READY** checks every resource, and **Smoke** publishes a test moment and waits for its call (one Gemini request).

In [ ]:
import base64
import json
import time

import google.auth
from google.auth.transport.requests import AuthorizedSession
from google.cloud import bigquery

# The region and BigQuery location from infra/variables.tf (var.region and
# var.dataset_location). Change them here if you changed them there.
REGION = "us-central1"
BQ_LOCATION = "US"

credentials, PROJECT_ID = google.auth.default(
    scopes=["https://www.googleapis.com/auth/cloud-platform"]
)
session = AuthorizedSession(credentials)
bq = bigquery.Client(project=PROJECT_ID, location=BQ_LOCATION)


def run_sql(sql):
    """Run a query from this notebook, with the project filled in, and return a DataFrame."""
    return bq.query(sql.replace("${project_id}", PROJECT_ID)).to_dataframe()


print("Project:", PROJECT_ID)

# The race in infra/variables.tf (var.openf1_session_key).
SESSION_KEY = "9693"

## Expect READY

In [ ]:
checks = {
    "race data in the bucket": f"https://storage.googleapis.com/storage/v1/b/{PROJECT_ID}-race-data/o/openf1%2F{SESSION_KEY}%2Fcar_data.json",
    "car photos in the bucket": f"https://storage.googleapis.com/storage/v1/b/{PROJECT_ID}-race-data/o/media%2Fcars%2Fmclaren.jpg",
    **{
        f"topic {t}": f"https://pubsub.googleapis.com/v1/projects/{PROJECT_ID}/topics/{t}"
        for t in ["race-events", "race-moments", "race-events-dead-letter"]
    },
    **{
        f"subscription {s}": f"https://pubsub.googleapis.com/v1/projects/{PROJECT_ID}/subscriptions/{s}"
        for s in [
            "race-events-bq",
            "race-moments-bq",
            "race-moments-strategist",
            "race-events-demo",
            "race-moments-demo",
        ]
    },
    "replayer job": f"https://run.googleapis.com/v2/projects/{PROJECT_ID}/locations/{REGION}/jobs/replayer",
}
for table in [
    "raw.raw_events",
    "serving.moments",
    "serving.calls",
    "raw.car_photos",
    "serving.cars",
]:
    bq.get_table(f"{PROJECT_ID}.{table}")
    print("  ok    table", table)
read_sql = (
    f"SELECT COUNT(*) AS n FROM `{PROJECT_ID}.serving.cars` "
    "WHERE status = '' AND ARRAY_LENGTH(sponsors) > 0"
)
read = next(iter(bq.query(read_sql).result()))["n"]
assert read >= 10, f"only {read} car photos have a reading"
print("  ok    a reading for every car photo")
for label, url in checks.items():
    session.get(url).raise_for_status()
    print("  ok   ", label)
for service in ["strategist", "pitwall"]:
    status = session.get(
        f"https://run.googleapis.com/v2/projects/{PROJECT_ID}/locations/{REGION}/services/{service}"
    ).json()
    ready = status.get("terminalCondition", {}).get("state") == "CONDITION_SUCCEEDED"
    print("  ok   " if ready else "  FAIL ", "service", service, "ready")
    assert ready, f"service {service} is not ready"
print("READY")

## Smoke

Publish a test `SAFETY_CAR` moment to `race-moments` and wait for the strategist's call in `serving.calls`. The test moment borrows the latest replay's `run_id`, so the strategist has a field to snapshot.

In [ ]:
latest = list(
    bq.query(
        f"SELECT run_id FROM `{PROJECT_ID}.raw.raw_events` WHERE event_type = 'drivers' ORDER BY publish_time DESC LIMIT 1"
    ).result()
)
run_id = latest[0]["run_id"] if latest else f"smoke-{int(time.time())}"
event_ts = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
moment = {
    "run_id": run_id,
    "moment": "SAFETY_CAR",
    "driver_number": None,
    "lap": 0,
    "event_ts": event_ts,
    "detail": "smoke test",
}
session.post(
    f"https://pubsub.googleapis.com/v1/projects/{PROJECT_ID}/topics/race-moments:publish",
    json={"messages": [{"data": base64.b64encode(json.dumps(moment).encode()).decode()}]},
).raise_for_status()
print("Published a test moment for", run_id)

config = bigquery.QueryJobConfig(
    query_parameters=[
        bigquery.ScalarQueryParameter("run_id", "STRING", run_id),
        bigquery.ScalarQueryParameter("event_ts", "TIMESTAMP", event_ts),
    ]
)
for _ in range(18):
    time.sleep(10)
    rows = list(
        bq.query(
            f"SELECT status, error FROM `{PROJECT_ID}.serving.calls` "
            "WHERE run_id = @run_id AND event_ts = @event_ts AND moment = 'SAFETY_CAR'",
            job_config=config,
        ).result()
    )
    if rows:
        assert rows[0]["status"] == "OK", f"the strategist wrote a FAILED call: {rows[0]['error']}"
        print("SMOKE OK")
        break
else:
    raise RuntimeError("no call within 3 minutes: check the strategist logs in Cloud Run")